In [1]:
import os
import re
import json
import time
from pathlib import Path

import torch
import pandas as pd
from datasets import load_dataset
from PIL import Image
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# =========================
# 1. CONFIGURATION
# =========================
MODEL_NAME = "/home/drnguyenvinh/Exam-Assistant/Bench_mark/Qwen3-VL-4B-Instruct"  
DATASET_NAME = "mm-eval/WebSRC"  
DATASET_SPLIT = "dev"
NUM_SAMPLES = 200
RANDOM_SEED = 42

MAX_NEW_TOKENS = 32
TEMPERATURE = 0.0
MAX_MODEL_LEN = 4096
GPU_MEM_UTIL = 0.55

RUN_NAME = "qwen3vl4b_websrc_benchmark"
OUTPUTDIR = Path(f"./benchmark_outputs/{RUN_NAME}_{time.strftime('%Y%m%d_%H%M%S')}")
OUTPUTDIR.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Outputs will be saved to: {OUTPUTDIR.resolve()}")

/opt/miniconda3/envs/qwen3_bm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Outputs will be saved to: /home/drnguyenvinh/Exam-Assistant/Bench_mark/benchmark_outputs/qwen3vl4b_websrc_benchmark_20260703_211933


In [2]:
# =========================
# 2. HELPER FUNCTIONS (SHORT-ANSWER OPTIMIZED)
# =========================
import re
import pynvml

def build_mm_prompt(tokenizer, question):
    """
    Sử dụng kỹ thuật Few-shot mô phỏng bằng chữ để ép Model hiểu 
    đầu ra chỉ được phép là một thực thể/con số duy nhất.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise Visual Question Answering engine. Your task is to extract "
                "the exact, minimal text or value from the image to answer the question. "
                "Strictly follow the output format constraints."
            )
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text", 
                    "text": (
                        f"Question: {question}\n\n"
                        f"STRICT OUTPUT RULES:\n"
                        f"- Output ONLY the exact short answer (e.g., a single location name, a number, a code, or a short value phrase).\n"
                        f"- Absolutely NO full sentences. NO explanations. NO reasoning steps.\n"
                        f"- NO introductory phrases like 'The answer is...', 'Based on the image...', or 'Looking at...'.\n\n"
                        f"EXAMPLES OF ALLOWED FORMAT:\n"
                        f"Tokyo\n"
                        f"1.94 USD\n"
                        f"38\n\n"
                        f"Final Answer:"
                    )
                },
            ],
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def normalize_text(s):
    if s is None:
        return ""
    s = str(s).strip().lower()
    
    # --- PHÒNG THỦ CHỦ ĐỘNG (Hậu xử lý xóa bỏ văn nói) ---
    # Nếu model cố tình trả lời dài dòng, ta sẽ gọt bớt các tiền tố phổ biến
    prefixes_to_strip = [
        "the answer is", "based on the image", "according to the image", 
        "looking at the", "it is", "the city is", "the cost is", "the number is"
    ]
    for prefix in prefixes_to_strip:
        if s.startswith(prefix):
            s = s[len(prefix):].strip()
            
    # Loại bỏ dấu chấm câu ở cuối từ nếu model quen tay thêm vào (ví dụ: "tokyo." -> "tokyo")
    s = re.sub(r'[.,;:!?]$', '', s)
    
    # Chuẩn hóa ký tự nháy kép đặc thù của WebSRC/Infographics
    s = s.replace('\\"', '"').replace('\\', '')
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def exact_match(pred, gold):
    if gold is None:
        return None
    if isinstance(gold, list):
        return int(any(normalize_text(pred) == normalize_text(g) for g in gold))
    return int(normalize_text(pred) == normalize_text(gold))

def get_current_gpu_usage(gpu_index=0):
    try:
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(gpu_index)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        pynvml.nvmlShutdown()
        return mem_info.used / (1024 ** 3)
    except Exception as e:
        print(f"[WARNING] Không thể đo VRAM qua pynvml: {e}")
        return 0.0

In [3]:
# =========================
# 3. PREPARE DATASET (FIXED FOR RAM LEAK)
# =========================
import json
from datasets import load_dataset
from transformers import AutoTokenizer

print(f"[INFO] Đang streaming và chuẩn hóa dữ liệu {DATASET_NAME}...")

dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

samples = []
vllm_inputs = []

for sample in dataset:
    # 🔴 CHỐT CHẶN CHỐNG TRÀN RAM: Đủ 200 mẫu (NUM_SAMPLES) là ngắt ngay!
    if len(samples) >= NUM_SAMPLES:
        break

    try:
        # 1. Giải mã JSON
        messages_str = sample.get('messages', '')
        if not messages_str:
            continue
        messages_list = json.loads(messages_str)
        
        if not messages_list or len(messages_list) == 0:
            continue
            
        # 2. Trích xuất Text
        qa_pair = messages_list[0]
        question = qa_pair.get('question', '').strip()
        ground_truth = qa_pair.get('answer', '').strip()
        
        # 3. Trích xuất Ảnh
        images = sample.get('media', [])
        if not images or len(images) == 0:
            continue
        image = images[0]  
        
        # 4. Định dạng prompt theo format của Qwen-VL
        prompt = build_mm_prompt(tokenizer, question)
        
        # 5. Lưu vào đúng biến mà Cell 4 cần
        samples.append({
            "id": sample["id"],
            "question": question,
            "gold_answer": ground_truth
        })
        
        vllm_inputs.append({
            "prompt": prompt,
            "multi_modal_data": {"image": image}
        })
        
    except Exception as e:
        print(f"[ERROR] Lỗi tại sample {sample.get('id', 'Unknown')}: {e}")
        continue

print(f"\n[SUCCESS] Đã load thành công! Số lượng sample hợp lệ: {len(samples)}")

[INFO] Đang streaming và chuẩn hóa dữ liệu mm-eval/WebSRC...

[SUCCESS] Đã load thành công! Số lượng sample hợp lệ: 200


In [4]:
# =========================
# 4. LOAD MODEL & INFERENCE
# =========================
# Chỉ nạp model vào GPU nếu đã load data thành công ở bước 3
if len(samples) > 0:
    print("[INFO] Initializing vLLM engine...")
    llm = LLM(
        model=MODEL_NAME,
        trust_remote_code=True,
        dtype="half",
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=GPU_MEM_UTIL,
        enforce_eager=True,          
        limit_mm_per_prompt={"image": 1},
    )

    sampling_params = SamplingParams(
        temperature=TEMPERATURE,
        max_tokens=MAX_NEW_TOKENS,
    )

    print("[INFO] Starting inference...")
    start_time = time.time()

    # ─── BƯỚC ĐO VRAM 1 ───
    peak_vram_gb = get_current_gpu_usage(gpu_index=0)

    # Cấu hình giới hạn pixel để tiết kiệm VRAM cho ảnh web/desktop
    mm_processor_kwargs = {
        "min_pixels": 256 * 256,
        "max_pixels": 1024 * 768  
    }

    outputs = llm.generate(
        vllm_inputs, 
        sampling_params=sampling_params,
        mm_processor_kwargs=mm_processor_kwargs
    )

    # ─── BƯỚC ĐO VRAM 2 ───
    current_vram = get_current_gpu_usage(gpu_index=0)
    if current_vram > peak_vram_gb:
        peak_vram_gb = current_vram

    total_time = time.time() - start_time
    print(f"🔥 ĐỈNH VRAM HỆ THỐNG THỰC TẾ ĐÃ DÙNG: {peak_vram_gb:.2f} GB")

    # =========================
    # 5. METRICS & SAVING
    # =========================
    predictions = [out.outputs[0].text.strip() for out in outputs]

    rows = []
    for s, pred in zip(samples, predictions):
        em = exact_match(pred, s["gold_answer"])
        rows.append({
            "id": s["id"],
            "question": s["question"],
            "gold": s["gold_answer"],
            "prediction": pred,
            "exact_match": em
        })

    df = pd.DataFrame(rows)

    # Cơ chế phòng ngừa lỗi DataFrame rỗng
    if df.empty:
        print("🚨 LỖI: DataFrame rỗng! Quá trình inference không trả về kết quả.")
    else:
        valid_em = df["exact_match"].dropna()
        overall_em = valid_em.mean() * 100 if len(valid_em) else 0.0
        avg_pred_len_words = df["prediction"].apply(lambda x: len(normalize_text(x).split())).mean()

        print("\n" + "="*30)
        print("🎯 EVALUATION RESULTS")
        print("="*30)
        print(f"Total Samples Processed : {len(df)}")
        print(f"Total Inference Time    : {total_time:.2f} seconds")
        print(f"Throughput              : {len(df) / total_time:.2f} samples/sec")
        print(f"Average Latency         : {total_time / len(df):.4f} sec/sample")
        print(f"Average Pred Length     : {avg_pred_len_words:.2f} words")
        print(f"Exact Match (EM)        : {overall_em:.2f}%")
        print("="*30)

        # Save Outputs
        csv_path = OUTPUTDIR / "predictions.csv"
        json_path = OUTPUTDIR / "predictions.jsonl"
        df.to_csv(csv_path, index=False)
        df.to_json(json_path, orient="records", lines=True)
        print(f"[INFO] Results saved to {OUTPUTDIR}")

else:
    print("🚨 Hệ thống đã tự động dừng ở Phần 3 vì không load được bất kỳ sample dữ liệu nào.")

[INFO] Initializing vLLM engine...
INFO 07-03 21:19:58 [utils.py:240] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.55, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 1}, 'model': '/home/drnguyenvinh/Exam-Assistant/Bench_mark/Qwen3-VL-4B-Instruct'}
INFO 07-03 21:19:58 [model.py:568] Resolved architecture: Qwen3VLForConditionalGeneration
WARNING 07-03 21:19:58 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 07-03 21:19:58 [model.py:1697] Using max model len 4096


2026-07-03 21:19:58,696	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 07-03 21:19:58 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-03 21:19:58 [vllm.py:886] Asynchronous scheduling is enabled.
WARNING 07-03 21:19:58 [vllm.py:942] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 07-03 21:19:58 [vllm.py:960] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 07-03 21:19:58 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 07-03 21:19:58 [vllm.py:1135] Cudagraph is disabled under eager mode
INFO 07-03 21:19:58 [compilation.py:303] Enabled custom fusions: norm_quant, act_quant


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=3409081) INFO 07-03 21:20:01 [core.py:109] Initializing a V1 LLM engine (v0.21.0) with config: model='/home/drnguyenvinh/Exam-Assistant/Bench_mark/Qwen3-VL-4B-Instruct', speculative_config=None, tokenizer='/home/drnguyenvinh/Exam-Assistant/Bench_mark/Qwen3-VL-4B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observabi

...

(EngineCore pid=3409081) INFO 07-03 21:20:04 [default_loader.py:397] Loading weights took 1.50 seconds
(EngineCore pid=3409081) INFO 07-03 21:20:04 [gpu_model_runner.py:4959] Model loading took 8.69 GiB memory and 1.644844 seconds
(EngineCore pid=3409081) INFO 07-03 21:20:04 [gpu_model_runner.py:5920] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=3409081) INFO 07-03 21:20:09 [gpu_worker.py:462] Available KV cache memory: 2.21 GiB
(EngineCore pid=3409081) INFO 07-03 21:20:09 [kv_cache_utils.py:1710] GPU KV cache size: 16,096 tokens
(EngineCore pid=3409081) INFO 07-03 21:20:09 [kv_cache_utils.py:1711] Maximum concurrency for 4,096 tokens per request: 3.93x
(EngineCore pid=3409081) INFO 07-03 21:20:09 [kernel_warmup.py:44] Skipping FlashInfer autotune because it is disabled.
(EngineCore pid=3409081) INFO 07-03 21:20:09 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations d

Rendering prompts:   0%|          | 1/200 [00:04<14:50,  4.48s/it]

(EngineCore pid=3409081) WARNING 07-03 21:20:13 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=3409081) WARNING 07-03 21:20:13 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _bilinear_pos_embed_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=3409081) WARNING 07-03 21:20:14 [jit_monitor.py:103] Triton kernel JIT compilation during inference: rotary_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=3409081) WARNING 07-03 21:20:14 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _triton_mrope_forward. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 200/200 [00:01<00:00, 116.17it/s, est. speed input: 100148.19 toks/s, output: 748.37 toks/s]

🔥 ĐỈNH VRAM HỆ THỐNG THỰC TẾ ĐÃ DÙNG: 13.56 GB

🎯 EVALUATION RESULTS
Total Samples Processed : 200
Total Inference Time    : 6.60 seconds
Throughput              : 30.31 samples/sec
Average Latency         : 0.0330 sec/sample
Average Pred Length     : 3.10 words
Exact Match (EM)        : 83.00%
[INFO] Results saved to benchmark_outputs/qwen3vl4b_websrc_benchmark_20260703_211933
